# CI/CD Log Data Generation
This cell generates synthetic CI/CD logs to train the NLP failure classification model.

In [1]:
import pandas as pd
import numpy as np
import os

os.makedirs('data', exist_ok=True)

categories = [
    "Build failure", "Dependency issue", "Security issue", 
    "Configuration error", "Test failure", "Deployment error", "Success"
]

log_templates = {
    "Build failure": ["fatal error: file not found", "make: *** [all] Error 2", "cannot compile module", "syntax error in file"],
    "Dependency issue": ["npm ERR! missing dependency", "pip install failed", "ModuleNotFoundError", "version conflict for package"],
    "Security issue": ["vulnerability found in package", "gitleaks: secret detected", "CORS policy violation", "unauthorized access token"],
    "Configuration error": ["YAML syntax error", "env variable missing", "invalid configuration file", "dockerfile build error"],
    "Test failure": ["Expected true but got false", "Test suite failed", "AssertionError: expected 5 to equal 4", "Jest failed with 1 failing test"],
    "Deployment error": ["Kubernetes pod crashloopbackoff", "timeout waiting for service", "failed to pull docker image", "AWS credential invalid"],
    "Success": ["build successful", "deployed successfully", "0 vulnerabilities", "all tests passed"]
}

data = []
for _ in range(2000):
    cat = np.random.choice(categories)
    log_text = np.random.choice(log_templates[cat]) + f" [process ID: {np.random.randint(100, 999)}]"
    data.append({"log_output": log_text, "issue_category": cat})

df = pd.DataFrame(data)
df.to_csv('data/cicd_logs.csv', index=False)
print("Saved synthetic CI/CD logs to data/cicd_logs.csv")


Saved synthetic CI/CD logs to data/cicd_logs.csv


# Train Failure Classification NLP Model
This cell trains a TF-IDF + Random Forest pipeline to classify the failure logs.

In [2]:
import pandas as pd
import joblib
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

os.makedirs('models', exist_ok=True)

df = pd.read_csv('data/cicd_logs.csv')
X = df['log_output']
y = df['issue_category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Build an NLP pipeline: Vectorizer -> Classifier
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=500)),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])

print("Training NLP Failure Classification Model...")
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print("Classification Report:\n", classification_report(y_test, y_pred))

joblib.dump(pipeline, 'models/failure_classifier.joblib')
print("Model saved to models/failure_classifier.joblib")


Training NLP Failure Classification Model...
Accuracy: 1.00
Classification Report:
                      precision    recall  f1-score   support

      Build failure       1.00      1.00      1.00        48
Configuration error       1.00      1.00      1.00        60
   Dependency issue       1.00      1.00      1.00        66
   Deployment error       1.00      1.00      1.00        60
     Security issue       1.00      1.00      1.00        57
            Success       1.00      1.00      1.00        52
       Test failure       1.00      1.00      1.00        57

           accuracy                           1.00       400
          macro avg       1.00      1.00      1.00       400
       weighted avg       1.00      1.00      1.00       400

Model saved to models/failure_classifier.joblib
